## 1. Carga de Datos

In [7]:
import pandas as pd

RAW_PATH = "Dataset_final.pkl"  # ajusta a tu ruta

def build_character_long_df(df: pd.DataFrame) -> pd.DataFrame:
    """Una fila por personaje, con su texto SIN modificar (sin quitar palabras)."""
    rows = []
    for _, row in df.iterrows():
        script = row["Script_Dict"] or {}
        genders = row["Characters_Genders"] or {}
        for character, text in script.items():
            if not text or not text.strip():
                continue
            gender = genders.get(character, "unknown")
            rows.append({
                "IMDb_ID": row["IMDb_ID"], "Title": row["Title"], "Award": row["Award"],
                "Character": character, "Gender": gender, "Text": text,
            })
    return pd.DataFrame(rows)

def load_all():
    df = pd.read_pickle(RAW_PATH)

    # Excluir películas de habla no inglesa
    peliculas_excluir = ["Talk to Her", "Anatomy of a Fall"]
    df = df[~df["Title"].isin(peliculas_excluir)].copy()

    # Películas únicas (dedup por IMDb_ID) -> para el gráfico general
    df_unique_movies = df.drop_duplicates(subset="IMDb_ID", keep="first")
    long_unique = build_character_long_df(df_unique_movies)

    # Con repeticiones a propósito -> para el gráfico por Award
    long_by_award = build_character_long_df(df)

    long_unique = long_unique[long_unique["Gender"].isin(["male", "female"])].copy()
    long_by_award = long_by_award[long_by_award["Gender"].isin(["male", "female"])].copy()

    return long_unique, long_by_award

long_unique, long_by_award = load_all()

## 2. Polaridad vs Subjetividad (TextBlob)

In [9]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from textblob import TextBlob

COLOR_MAP = {"female": "#e0a72e", "male": "#9e9e9e"}
LABEL_MAP = {"female": "Femenino", "male": "Masculino"}
GRID_COLOR = "rgba(0,0,0,0.08)"

def compute_sentiment_textblob(long_df: pd.DataFrame) -> pd.DataFrame:
    long_df = long_df.copy()
    blobs = long_df["Text"].apply(TextBlob)  # texto crudo, sin quitar palabras
    long_df["Polarity"] = blobs.apply(lambda b: b.sentiment.polarity)
    long_df["Subjectivity"] = blobs.apply(lambda b: b.sentiment.subjectivity)
    return long_df

def aggregate_by_gender(long_df: pd.DataFrame, group_cols):
    return (
        long_df.groupby(group_cols)
        .agg(Polaridad=("Polarity", "mean"),
             Subjetividad=("Subjectivity", "mean"),
             N_Personajes=("Character", "count"))
        .reset_index()
    )

def make_bubble_fig(agg: pd.DataFrame, title: str):
    """Bubble chart estilo claro/estándar (fondo blanco), igual a tu imagen de referencia."""
    fig = go.Figure()
    max_n = agg["N_Personajes"].max()
    sizeref = 2.0 * max_n / (46.0 ** 2)

    for gender in ["male", "female"]:
        sub = agg[agg["Gender"] == gender]
        if sub.empty:
            continue
        fig.add_trace(go.Scatter(
            x=sub["Polaridad"], y=sub["Subjetividad"], mode="markers",
            name=LABEL_MAP[gender],
            marker=dict(size=sub["N_Personajes"], sizemode="area", sizeref=sizeref,
                        sizemin=10, color=COLOR_MAP[gender],
                        line=dict(width=1, color="rgba(0,0,0,0.25)"), opacity=0.9),
            customdata=sub[["N_Personajes"]],
            hovertemplate=("<b>%{fullData.name}</b><br>Polaridad: %{x:.3f}<br>"
                          "Subjetividad: %{y:.3f}<br>Nº personajes: %{customdata[0]}<extra></extra>"),
        ))

    fig.update_layout(
        template="plotly_white",
        title=dict(text=title, font=dict(size=16, color="#1a1a1a")),
        xaxis=dict(title="Polaridad", gridcolor=GRID_COLOR, zeroline=False),
        yaxis=dict(title="Subjetividad", gridcolor=GRID_COLOR, zeroline=False),
        legend=dict(title="Gender"),
        margin=dict(l=60, r=40, t=60, b=50),
    )
    return fig

# --- Gráfico general (películas únicas) ---
lu = compute_sentiment_textblob(long_unique)
agg_general = aggregate_by_gender(lu, ["Gender"])
fig1 = make_bubble_fig(agg_general, "Polaridad vs Subjetividad media por sexo")
fig1.show()

## 3. Versión por Award 

In [10]:
def make_bubble_fig_by_award_stacked(agg: pd.DataFrame, title: str):
    """Un panel por Award, apilados EN VERTICAL (facet_row), no en columnas."""
    awards = sorted(agg["Award"].unique())
    max_n = agg["N_Personajes"].max()
    sizeref = 2.0 * max_n / (40.0 ** 2)

    fig = make_subplots(
        rows=len(awards), cols=1,
        subplot_titles=awards,
        shared_xaxes=True,
        vertical_spacing=0.10,
    )

    for i, award in enumerate(awards, start=1):
        for gender in ["male", "female"]:
            sub = agg[(agg["Award"] == award) & (agg["Gender"] == gender)]
            if sub.empty:
                continue
            fig.add_trace(go.Scatter(
                x=sub["Polaridad"], y=sub["Subjetividad"], mode="markers",
                name=LABEL_MAP[gender], legendgroup=gender, showlegend=(i == 1),
                marker=dict(size=sub["N_Personajes"], sizemode="area", sizeref=sizeref,
                            sizemin=10, color=COLOR_MAP[gender],
                            line=dict(width=1, color="rgba(0,0,0,0.25)"), opacity=0.9),
                customdata=sub[["N_Personajes"]],
                hovertemplate=("<b>%{fullData.name}</b><br>Polaridad: %{x:.3f}<br>"
                              "Subjetividad: %{y:.3f}<br>Nº personajes: %{customdata[0]}<extra></extra>"),
            ), row=i, col=1)
        fig.update_yaxes(title_text="Subjetividad", gridcolor=GRID_COLOR, row=i, col=1)

    fig.update_xaxes(title_text="Polaridad", gridcolor=GRID_COLOR, row=len(awards), col=1)
    fig.update_layout(
        template="plotly_white",
        title=dict(text=title, font=dict(size=16, color="#1a1a1a")),
        legend=dict(title="Gender"),
        height=300 * len(awards),
        margin=dict(l=60, r=40, t=80, b=50),
    )
    return fig

lba = compute_sentiment_textblob(long_by_award)
agg_award = aggregate_by_gender(lba, ["Award", "Gender"])
fig2 = make_bubble_fig_by_award_stacked(agg_award, "Polaridad vs Subjetividad media por sexo y Award")
fig2.show()

## 4. Análisis de sentimiento con VADER + spaCy (frase a frase)

In [11]:
import spacy
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# Solo detección de frases, sin modelo estadístico -> rápido y no requiere
# lematización/stopwords (VADER necesita el texto crudo, frase a frase)
_nlp = spacy.blank("en")
_nlp.add_pipe("sentencizer")
_sia = SentimentIntensityAnalyzer()

def split_into_sentences(text: str):
    doc = _nlp(text)
    return [s.text.strip() for s in doc.sents if s.text.strip()]

def compute_sentence_level_sentiment(long_df: pd.DataFrame) -> pd.DataFrame:
    records = []
    for _, row in long_df.iterrows():
        for sent in split_into_sentences(row["Text"]):
            scores = _sia.polarity_scores(sent)
            records.append({
                "IMDb_ID": row["IMDb_ID"], "Award": row["Award"], "Character": row["Character"],
                "Gender": row["Gender"], "Compound": scores["compound"], "Pos": scores["pos"],
            })
    return pd.DataFrame(records)

def aggregate_vader_by_gender(sent_df: pd.DataFrame, group_cols):
    # Importante: agrupar también por IMDb_ID + Character antes de promediar,
    # para no mezclar personajes con el mismo nombre de películas distintas
    per_char = (
        sent_df.groupby(["IMDb_ID", "Character"] + group_cols)
        .agg(Compound=("Compound", "mean"), Pos=("Pos", "mean"))
        .reset_index()
    )
    return (
        per_char.groupby(group_cols)
        .agg(Compound_medio=("Compound", "mean"),
             Positividad_media=("Pos", "mean"),
             N_Personajes=("Compound", "count"))
        .reset_index()
    )

def make_vader_bubble_fig(agg: pd.DataFrame, title: str):
    fig = go.Figure()
    max_n = agg["N_Personajes"].max()
    sizeref = 2.0 * max_n / (46.0 ** 2)

    for gender in ["male", "female"]:
        sub = agg[agg["Gender"] == gender]
        if sub.empty:
            continue
        fig.add_trace(go.Scatter(
            x=sub["Compound_medio"], y=sub["Positividad_media"], mode="markers",
            name=LABEL_MAP[gender],
            marker=dict(size=sub["N_Personajes"], sizemode="area", sizeref=sizeref,
                        sizemin=10, color=COLOR_MAP[gender],
                        line=dict(width=1, color="rgba(0,0,0,0.25)"), opacity=0.9),
            customdata=sub[["N_Personajes"]],
            hovertemplate=("<b>%{fullData.name}</b><br>Compound: %{x:.3f}<br>"
                          "Positividad media: %{y:.3f}<br>Nº personajes: %{customdata[0]}<extra></extra>"),
        ))

    fig.update_layout(
        template="plotly_white",
        title=dict(text=title, font=dict(size=16, color="#1a1a1a")),
        xaxis=dict(title="Compound medio (VADER)", gridcolor=GRID_COLOR, zeroline=False),
        yaxis=dict(title="Positividad media (VADER)", gridcolor=GRID_COLOR, zeroline=False),
        legend=dict(title="Gender"),
        margin=dict(l=60, r=40, t=60, b=50),
    )
    return fig

# --- General (películas únicas) ---
sent_unique = compute_sentence_level_sentiment(long_unique)
agg_vader_general = aggregate_vader_by_gender(sent_unique, ["Gender"])
fig3 = make_vader_bubble_fig(agg_vader_general, "Sentimiento medio (VADER) por sexo")
fig3.show()

# --- Por Award (apiladas en vertical, misma función de stacking) ---
sent_award = compute_sentence_level_sentiment(long_by_award)
agg_vader_award = aggregate_vader_by_gender(sent_award, ["Award", "Gender"])
fig4 = make_bubble_fig_by_award_stacked(  # reutilizamos la misma función; solo cambia qué columnas se pasan
    agg_vader_award.rename(columns={"Compound_medio": "Polaridad", "Positividad_media": "Subjetividad"}),
    "Sentimiento medio (VADER) por sexo y Award"
)
fig4.show()